In [1]:
!pip install -q vllm
!pip uninstall -y torchaudio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:


In [2]:
!git clone https://github.com/VatsalRoy/keep-it-warm.git
%cd /kaggle/working/keep-it-warm
!python -m env.generate_world --seed 42
!ls runs/baseline_sessions

Cloning into 'keep-it-warm'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (208/208), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 208 (delta 94), reused 164 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (208/208), 337.80 KiB | 3.04 MiB/s, done.
Resolving deltas: 100% (94/94), done.
/kaggle/working/keep-it-warm
seed=42  world_sha256=4a5b9a1a8c73a5e39f83a7c00eb5d4ccafec0d2bf6912ce82e81f31f2f0dc419
episodes.jsonl	turns.jsonl


In [3]:
import os, sys, subprocess, time, requests

def start_server(gpu, port):
    cmd = ["vllm", "serve", "Qwen/Qwen3-4B-Instruct-2507", "--dtype", "half",
           "--max-model-len", "28672", "--enable-prefix-caching", "--enable-prompt-tokens-details",
           "--gpu-memory-utilization", "0.90", "--port", str(port)]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "VLLM_SERVER_DEV_MODE": "1"}
    return subprocess.Popen(cmd, stdout=open(f"vllm_gpu{gpu}.log", "w"), stderr=subprocess.STDOUT, env=env)

def wait_ready(proc, port, log):
    for i in range(120):
        if proc.poll() is not None:
            print(f"port {port} CRASHED"); os.system(f"tail -n 30 {log}"); return False
        try:
            if requests.get(f"http://localhost:{port}/health", timeout=2).status_code == 200:
                print(f"port {port} READY after ~{i * 10}s"); return True
        except Exception:
            pass
        time.sleep(10)
    print(f"port {port} not ready after 20 min"); return False

s0 = start_server(0, 8000); assert wait_ready(s0, 8000, "vllm_gpu0.log")
s1 = start_server(1, 8001); assert wait_ready(s1, 8001, "vllm_gpu1.log")
for port in (8000, 8001):
    print(port, "reset ->", requests.post(f"http://localhost:{port}/reset_prefix_cache").status_code)

port 8000 READY after ~220s
port 8001 READY after ~120s
8000 reset -> 200
8001 reset -> 200


In [4]:
!python scripts/run_episodes.py --policy summary --threshold 8192 --out runs/main/P4_summary8k_v2 --base http://localhost:8000

s01            s0 4/4 ✓✓✓✓  turns=20  end=completed        peak_ctx=6961   cached=94% $B=0.1079 wall=105.9s
s02            s0 2/4 ✓✓✗✗  turns=23  end=completed        peak_ctx=11349  cached=89% $B=0.1490 wall=170.0s
s03            s0 1/4 ✗✗✗✓  turns=38  end=completed        peak_ctx=9696   cached=89% $B=0.2100 wall=235.4s
s04            s0 0/4 ✗✗✗✗  turns=20  end=completed        peak_ctx=8381   cached=86% $B=0.1420 wall=136.3s
s05            s0 2/4 ✗✓✓✗  turns=27  end=completed        peak_ctx=7306   cached=94% $B=0.1168 wall=112.9s
s06            s0 1/4 ✗✗✗✓  turns=12  end=completed        peak_ctx=7150   cached=88% $B=0.0843 wall=92.4s
s07            s0 1/4 ✓✗✗✗  turns=25  end=completed        peak_ctx=7244   cached=97% $B=0.1682 wall=242.9s
s08            s0 2/4 ✗✓✓✗  turns=27  end=completed        peak_ctx=8291   cached=91% $B=0.1861 wall=229.4s
s09            s0 1/4 ✗✗✗✓  turns=46  end=completed        peak_ctx=7687   cached=98% $B=0.2251 wall=253.1s
s10            s0 1/4 ✗✗✓✗  t